# NLT Replication: Visual Analysis

Purpose: visualize accuracy, variance, errors, token usage, and NLT gains.
- Loads aggregated_results.csv with token usage columns
- Summarizes by approach, model, scenario, and perturbation
- Generates charts for the replication study
- Saves PNGs to results/figures/

Run order: install deps → load data → summaries → plots

In [ ]:
# Optional dependency install
import importlib.util
import subprocess
import sys

def ensure(pkg: str):
    if importlib.util.find_spec(pkg) is None:
        print(f'Installing {pkg} ...')
        subprocess.check_call(['uv', 'pip', 'install', pkg])
    else:
        print(f'{pkg} already available')

for pkg in ['pandas', 'matplotlib', 'seaborn']:
    ensure(pkg)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

ROOT = Path.cwd()
csv_candidates = [ROOT / 'aggregated_results.csv', ROOT.parent / 'aggregated_results.csv']
for candidate in csv_candidates:
    if candidate.exists():
        CSV_PATH = candidate
        break
else:
    raise FileNotFoundError('aggregated_results.csv not found')

FIG_DIR = ROOT / 'results' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Using CSV:', CSV_PATH)
print('Saving figures to:', FIG_DIR)
sns.set_theme(style='whitegrid', palette='colorblind')

In [ ]:
# Load data
df = pd.read_csv(CSV_PATH)

# Convert numeric columns
num_cols = ['accuracy', 'variance', 'total', 'errors', 'valid_trials', 'total_tokens', 'prompt_tokens', 'completion_tokens']
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

df['error_rate'] = df['errors'] / df['total'].where(df['total'] > 0, 1)
df['success'] = (df['accuracy'] > 0) & (df['error_rate'] <= 0.75)

print(f"Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")
df.head()

## Summary Statistics

In [ ]:
# Summary by approach
by_approach = df.groupby('approach').agg({
    'accuracy': 'mean',
    'variance': 'mean',
    'errors': 'sum',
    'total_tokens': 'sum',
    'prompt_tokens': 'sum',
    'completion_tokens': 'sum'
}).round(4)

print("\n=== BY APPROACH ===")
print(by_approach)

In [ ]:
# Summary by model and approach
by_model = df.groupby(['model_id', 'approach']).agg({
    'accuracy': 'mean',
    'errors': 'sum'
}).round(4)

print("\n=== BY MODEL & APPROACH ===")
print(by_model)

## Visualizations

In [ ]:
# Figure 1: Accuracy by Approach
fig, ax = plt.subplots(figsize=(8, 5))
approach_acc = df.groupby('approach')['accuracy'].mean()
approach_acc.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Mean Accuracy by Approach', fontsize=14, fontweight='bold')
ax.set_xlabel('Approach')
ax.set_ylabel('Accuracy')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig1_accuracy_by_approach.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 2: Accuracy by Model (NLT vs Structured)
model_pivot = df.pivot_table(index='model_id', columns='approach', values='accuracy', aggfunc='mean')
model_pivot = model_pivot.sort_values('nlt', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
model_pivot.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Accuracy by Model: NLT vs Structured', fontsize=14, fontweight='bold')
ax.set_xlabel('Model')
ax.set_ylabel('Accuracy')
ax.set_xticklabels([m.split('/')[-1][:20] for m in model_pivot.index], rotation=45, ha='right')
ax.legend(title='Approach')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_accuracy_by_model.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 3: Error Counts by Approach
fig, ax = plt.subplots(figsize=(8, 5))
error_counts = df.groupby('approach')['errors'].sum()
error_counts.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Total Errors by Approach', fontsize=14, fontweight='bold')
ax.set_xlabel('Approach')
ax.set_ylabel('Total Errors')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_errors_by_approach.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 4: Variance Comparison
fig, ax = plt.subplots(figsize=(8, 5))
variance_data = df.groupby('approach')['variance'].mean()
variance_data.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Mean Variance by Approach', fontsize=14, fontweight='bold')
ax.set_xlabel('Approach')
ax.set_ylabel('Variance')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig4_variance_by_approach.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 5: Accuracy by Scenario
scenario_pivot = df.pivot_table(index='scenario', columns='approach', values='accuracy', aggfunc='mean')

fig, ax = plt.subplots(figsize=(8, 5))
scenario_pivot.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Accuracy by Scenario: NLT vs Structured', fontsize=14, fontweight='bold')
ax.set_xlabel('Scenario')
ax.set_ylabel('Accuracy')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Approach')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig5_accuracy_by_scenario.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 6: Perturbation Robustness
pert_pivot = df.pivot_table(index='perturbed', columns='approach', values='accuracy', aggfunc='mean')
pert_pivot.index = ['Non-perturbed', 'Perturbed']

fig, ax = plt.subplots(figsize=(8, 5))
pert_pivot.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_title('Perturbation Robustness: NLT vs Structured', fontsize=14, fontweight='bold')
ax.set_xlabel('Prompt Type')
ax.set_ylabel('Accuracy')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Approach')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig6_perturbation_robustness.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 7: Token Usage Comparison
token_data = df.groupby('approach')[['total_tokens', 'prompt_tokens', 'completion_tokens']].sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Total tokens
token_data['total_tokens'].plot(kind='bar', ax=ax1, color=['#2ecc71', '#e74c3c'])
ax1.set_title('Total Token Usage by Approach', fontsize=14, fontweight='bold')
ax1.set_xlabel('Approach')
ax1.set_ylabel('Total Tokens')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0)
ax1.grid(axis='y', alpha=0.3)

# Token breakdown
token_data[['prompt_tokens', 'completion_tokens']].plot(kind='bar', ax=ax2, stacked=True)
ax2.set_title('Token Breakdown by Approach', fontsize=14, fontweight='bold')
ax2.set_xlabel('Approach')
ax2.set_ylabel('Tokens')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=0)
ax2.legend(title='Token Type', labels=['Input', 'Output'])
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig7_token_usage.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 8: NLT Gains by Model
gains_df = df.pivot_table(index='model_id', columns='approach', values='accuracy', aggfunc='mean')
gains_df = gains_df.dropna(subset=['nlt', 'structured'])
gains_df['gain'] = gains_df['nlt'] - gains_df['structured']
gains_df = gains_df.sort_values('gain', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#e74c3c' if x < 0 else '#2ecc71' for x in gains_df['gain']]
gains_df['gain'].plot(kind='barh', ax=ax, color=colors)
ax.set_title('NLT Accuracy Gain over Structured (by Model)', fontsize=14, fontweight='bold')
ax.set_xlabel('Accuracy Gain (NLT - Structured)')
ax.set_ylabel('Model')
ax.set_yticklabels([m.split('/')[-1][:30] for m in gains_df.index])
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig8_nlt_gains_by_model.png', dpi=300, bbox_inches='tight')
plt.show()

## Complete

All figures saved to `results/figures/`